In [8]:
# Inter-Annotator Agreement (IAA) Analysis - Cross-Round Version
# Compares annotators across ALL rounds (ignores round number)
# Uses highest version number per case/annotator combination

import pandas as pd
from supabase import create_client, Client
import os

# ---------- Setup ----------
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# Fields to EXCLUDE from comparison
EXCLUDED_FIELDS = [
    # Reference fields (end with _ref)
    "complaint_agreement_ref",
    "declaration_agreement_ref",
    "ownership_chain_refs",
    "chargeoff_balance_ref",
    "substantiated_last_payment_ref",
    "chargeoff_creditor_ref",
    "debtor_info_ref",
    "post_charge_off_purchaser_refs",
    "fees_proof_ref",
    "decl_ref_1", "decl_ref_2", "decl_ref_3", "decl_ref_4", "decl_ref_5",
    "decl_ref_6", "decl_ref_7", "decl_ref_8", "decl_ref_9", "decl_ref_10",
    
    # Info fields (contain _info)
    "alleges_chargeoff_creditor_info_6",
    "alleged_chargeoff_creditor_info_6",
    "alleges_debtor_last_known_address_7",
    "alleged_debtor_last_known_address_7",
    "alleges_postchargeoffpurchaserinfo_0",
    "postchargeoffpurchaserinfo_0",
    "postchargeoffpurchaserinfo_1",
    
    # Final notes
    "final_notes",
    
    # System/metadata fields
    "case_number",
    "annotator_id",
    "version",
    "round",
    "created_at",
    "time_caselevel",
]

# All fields to compare
FIELDS_TO_COMPARE = [
    # Section 1
    "rfdj_demand_amount",
    "rfdj_interest_amount",
    "rfdj_cost_amount",
    "rfdj_attorney_fees_amount",
    "complaint_has_prayer",
    "rfdjamount_1",
    
    # Section 2
    "allegation_debt_buyer_1",
    "allegation_nature_of_debt_2",
    "allegation_sole_owner_3",
    "alleges_chargeoff_balance_4",
    "alleged_chargeoff_balance_4",
    "alleges_last_payment_date_5",
    "alleged_last_payment_date_5",
    "alleges_default_date_5b",
    "alleged_default_date_5b",
    "allegation_1788_52_compliance",
    
    # Section 3
    "hascontractorlaststatement_0",
    
    # Section 4 - Declarations
    "num_creditors_chain",
    "decl_exists_1", "decl_pkm_1", "decl_perjury_1", "decl_calaw_1",
    "decl_exists_2", "decl_pkm_2", "decl_perjury_2", "decl_calaw_2",
    "decl_exists_3", "decl_pkm_3", "decl_perjury_3", "decl_calaw_3",
    "decl_exists_4", "decl_pkm_4", "decl_perjury_4", "decl_calaw_4",
    "decl_exists_5", "decl_pkm_5", "decl_perjury_5", "decl_calaw_5",
    "decl_exists_6", "decl_pkm_6", "decl_perjury_6", "decl_calaw_6",
    "decl_exists_7", "decl_pkm_7", "decl_perjury_7", "decl_calaw_7",
    "decl_exists_8", "decl_pkm_8", "decl_perjury_8", "decl_calaw_8",
    "decl_exists_9", "decl_pkm_9", "decl_perjury_9", "decl_calaw_9",
    "decl_exists_10", "decl_pkm_10", "decl_perjury_10", "decl_calaw_10",
    
    # Section 5
    "declaration_has_agreement_proof",
    
    # Section 6
    "ownership_chain_sufficient",
    "ownership_chain_account_match",
    
    # Section 7-11
    "haschargeoffbalance_1",
    "lastpaymentdate_1",
    "substantiated_last_payment_date",
    "lastpaymentdate_2",
    "chargeoffcreditorinfo_1",
    "debtorinfo_1",
    
    # Section 12
    "fees_proof_present",
    
    # Final
    "final_recommendation",
]


def fetch_results_gold_all_rounds() -> pd.DataFrame:
    """Fetch all results from results_gold across ALL rounds."""
    res = (
        supabase.table("results_gold")
        .select("*")
        .order("case_number", desc=False)
        .order("annotator_id", desc=False)
        .order("version", desc=True)
        .execute()
    )
    
    df = pd.DataFrame(res.data)
    
    # Exclude annotator 'aviv'
    df = df[df['annotator_id'].str.lower() != 'aviv']
    
    # Exclude specific cases
    excluded_cases = ['23CHLC18998', '23CHLC18504', '23CHLC16737']
    df = df[~df['case_number'].isin(excluded_cases)]
    
    # Keep only HIGHEST version per case/annotator (ignoring round)
    df = df.sort_values(['case_number', 'annotator_id', 'version'], ascending=[True, True, False])
    df = df.drop_duplicates(subset=['case_number', 'annotator_id'], keep='first')
    
    return df


def normalize_value(value) -> str:
    """
    Normalize values for comparison.
    Handles formatting differences:
    - Numbers: 3000 vs 3,000 vs $3,000 vs 2173.33 vs $2,173.33
    - Dates: 8/12/2018 vs 08/12/2018 vs 2018-08-12
    - Empty/Zero: treats 0, "0", "0.0", "" as equivalent
    """
    import re
    
    if pd.isna(value) or value is None or value == "":
        return ""
    
    # Convert to string and strip whitespace
    val_str = str(value).strip()
    
    # Treat "0" or "0.0" as empty
    if val_str in ["0", "0.0", "0.00"]:
        return ""
    
    # Try to normalize as number (remove $, commas, spaces)
    # Remove common currency/formatting characters
    cleaned = val_str.replace('$', '').replace(',', '').replace(' ', '').lower()
    
    # Check if it's a number
    try:
        num_val = float(cleaned)
        # Treat 0 as empty
        if num_val == 0:
            return ""
        # Return as string with consistent format (removes trailing zeros)
        return str(num_val)
    except ValueError:
        pass  # Not a number, continue
    
    # Try to normalize as date (MM/DD/YYYY, M/D/YYYY, MM/DD/YY, YYYY-MM-DD)
    val_lower = val_str.lower()
    
    # Match patterns like 8/12/2018, 08/12/2018 (4-digit year)
    date_match = re.match(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})$', val_lower)
    if date_match:
        month, day, year = date_match.groups()
        # Normalize to MM/DD/YYYY
        return f"{int(month):02d}/{int(day):02d}/{year}"
    
    # Match patterns like 8/22/19, 08/22/19 (2-digit year)
    date_match = re.match(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{2})$', val_lower)
    if date_match:
        month, day, year = date_match.groups()
        # Convert 2-digit year to 4-digit (assume 2000s)
        full_year = f"20{year}"
        # Normalize to MM/DD/YYYY
        return f"{int(month):02d}/{int(day):02d}/{full_year}"
    
    # ISO format: YYYY-MM-DD
    date_match = re.match(r'^(\d{4})[/-](\d{1,2})[/-](\d{1,2})$', val_lower)
    if date_match:
        year, month, day = date_match.groups()
        # Normalize to MM/DD/YYYY
        return f"{int(month):02d}/{int(day):02d}/{year}"
    
    # Return as-is for text (lowercased and stripped)
    return val_lower


def compare_annotators(df: pd.DataFrame):
    """Compare all fields between annotators for each case (ignoring round)."""
    
    # Get cases that have exactly 2 annotators
    case_annotator_counts = df.groupby('case_number')['annotator_id'].nunique()
    cases_with_two_annotators = case_annotator_counts[case_annotator_counts == 2].index.tolist()
    
    print(f"Found {len(cases_with_two_annotators)} cases with exactly 2 annotators")
    
    if len(cases_with_two_annotators) == 0:
        print("\n⚠️ No cases with 2 annotators found!")
        print("\nAnnotators per case distribution:")
        print(case_annotator_counts.value_counts().sort_index())
        return pd.DataFrame(), pd.DataFrame()
    
    disagreements = []
    
    for case_num in cases_with_two_annotators:
        case_data = df[df['case_number'] == case_num]
        
        if len(case_data) != 2:
            continue
        
        annotators = case_data['annotator_id'].tolist()
        rounds = case_data['round'].tolist() if 'round' in case_data.columns else ['?', '?']
        
        ann1_data = case_data[case_data['annotator_id'] == annotators[0]].iloc[0]
        ann2_data = case_data[case_data['annotator_id'] == annotators[1]].iloc[0]
        
        for field in FIELDS_TO_COMPARE:
            if field not in ann1_data or field not in ann2_data:
                continue
            
            val1 = normalize_value(ann1_data.get(field))
            val2 = normalize_value(ann2_data.get(field))
            
            # Check for disagreement
            if val1 != val2:
                # Don't count as disagreement if both are empty
                if not (val1 == "" and val2 == ""):
                    disagreements.append({
                        'case_number': case_num,
                        'field': field,
                        'annotator_1': annotators[0],
                        'round_1': rounds[0],
                        'value_1': val1,
                        'annotator_2': annotators[1],
                        'round_2': rounds[1],
                        'value_2': val2,
                    })
    
    detailed_df = pd.DataFrame(disagreements)
    
    # Create summary by case
    if not detailed_df.empty:
        summary_df = detailed_df.groupby('case_number').agg({
            'field': 'count',
            'annotator_1': 'first',
            'annotator_2': 'first',
            'round_1': 'first',
            'round_2': 'first'
        }).reset_index()
        summary_df.columns = ['case_number', 'disagreement_count', 'annotator_1', 'annotator_2', 'round_1', 'round_2']
        summary_df = summary_df.sort_values('disagreement_count', ascending=False)
    else:
        summary_df = pd.DataFrame(columns=['case_number', 'disagreement_count', 'annotator_1', 'annotator_2', 'round_1', 'round_2'])
    
    return summary_df, detailed_df


# ===== RUN ANALYSIS =====

print("Fetching data from Supabase (all rounds)...")
df = fetch_results_gold_all_rounds()

print(f"Total records (after keeping highest version): {len(df)}")
print(f"Unique cases: {df['case_number'].nunique()}")
print(f"Unique annotators: {df['annotator_id'].nunique()}")

if 'round' in df.columns:
    print(f"\nRound distribution:")
    print(df.groupby(['round', 'annotator_id']).size().unstack(fill_value=0))

print("\nAnalyzing inter-annotator agreement...")
summary_df, detailed_df = compare_annotators(df)

# ===== STATISTICS =====

if not detailed_df.empty:
    num_cases = detailed_df['case_number'].nunique()
    total_comparisons = num_cases * len(FIELDS_TO_COMPARE)
    total_disagreements = len(detailed_df)
    agreement_rate = 1 - (total_disagreements / total_comparisons)
    
    print("\n" + "="*80)
    print("INTER-ANNOTATOR AGREEMENT STATISTICS")
    print("="*80)
    print(f"Fields compared: {len(FIELDS_TO_COMPARE)}")
    print(f"Cases with 2 annotators: {num_cases}")
    print(f"Total comparisons: {total_comparisons}")
    print(f"Total disagreements: {total_disagreements}")
    print(f"Agreement rate: {agreement_rate:.2%}")
    
    # Fields with most disagreements
    print("\nTop 15 fields with most disagreements:")
    field_counts = detailed_df['field'].value_counts().head(15)
    for field, count in field_counts.items():
        pct = (count / num_cases) * 100
        print(f"  {field}: {count} ({pct:.1f}% of cases)")
else:
    print("\n✅ Perfect agreement! No disagreements found.")

# ===== DISPLAY RESULTS =====

if not summary_df.empty:
    print("\n" + "="*80)
    print("CASES WITH DISAGREEMENTS (sorted by count)")
    print("="*80)
    display(summary_df.head(20))

    print("\n" + "="*80)
    print("SAMPLE DETAILED DISAGREEMENTS (first 50)")
    print("="*80)
    display(detailed_df.head(50))

    # ===== SAVE OUTPUTS =====

    output_dir = "/Users/othmanbensouda/Desktop/debt_collection_website/analysis_disagreement"
    os.makedirs(output_dir, exist_ok=True)

    summary_df.to_excel(f"{output_dir}/disagreement_summary.xlsx", index=False)
    detailed_df.to_excel(f"{output_dir}/detailed_disagreements.xlsx", index=False)

    # Pivot view: case x field matrix
    pivot_df = detailed_df.pivot_table(
        index='case_number',
        columns='field',
        aggfunc='size',
        fill_value=0
    )
    pivot_df.to_excel(f"{output_dir}/disagreement_matrix.xlsx")

    print(f"\n✅ Results saved to {output_dir}/")
    print(f"   - disagreement_summary.xlsx: Cases ranked by disagreement count")
    print(f"   - detailed_disagreements.xlsx: Field-level disagreements with values")
    print(f"   - disagreement_matrix.xlsx: Pivot view (cases × fields)")

    # ===== CASES NEEDING REVIEW =====

    print("\n" + "="*80)
    print("CASES NEEDING REVIEW (10+ disagreements)")
    print("="*80)
    high_disagreement = summary_df[summary_df['disagreement_count'] >= 10]
    if not high_disagreement.empty:
        display(high_disagreement)
        print(f"\n⚠️ {len(high_disagreement)} cases need immediate review")
    else:
        print("✅ No cases with 10+ disagreements")

    # ===== FIELD CATEGORIES WITH MOST ISSUES =====

    print("\n" + "="*80)
    print("FIELD CATEGORIES WITH MOST DISAGREEMENTS")
    print("="*80)
    
    def categorize_field(field):
        if field.startswith('decl_'):
            return 'Declarations'
        elif 'rfdj' in field:
            return 'RFDJ Amounts'
        elif 'allegation' in field or 'alleges' in field or 'alleged' in field:
            return 'Allegations'
        elif 'ownership' in field:
            return 'Ownership Chain'
        elif 'fees' in field:
            return 'Attorney Fees'
        elif 'final' in field:
            return 'Final Recommendation'
        else:
            return 'Other'
    
    detailed_df['category'] = detailed_df['field'].apply(categorize_field)
    category_counts = detailed_df['category'].value_counts()
    
    for category, count in category_counts.items():
        pct = (count / len(detailed_df)) * 100
        print(f"  {category}: {count} disagreements ({pct:.1f}%)")

Fetching data from Supabase (all rounds)...
Total records (after keeping highest version): 208
Unique cases: 104
Unique annotators: 3

Round distribution:
annotator_id  Brian  Parker  Victor
round                              
1                34      35      35
2                34      34      36

Analyzing inter-annotator agreement...
Found 104 cases with exactly 2 annotators

INTER-ANNOTATOR AGREEMENT STATISTICS
Fields compared: 69
Cases with 2 annotators: 101
Total comparisons: 6969
Total disagreements: 820
Agreement rate: 88.23%

Top 15 fields with most disagreements:
  decl_perjury_2: 70 (69.3% of cases)
  alleges_default_date_5b: 64 (63.4% of cases)
  decl_pkm_2: 64 (63.4% of cases)
  decl_calaw_2: 61 (60.4% of cases)
  fees_proof_present: 49 (48.5% of cases)
  decl_exists_2: 43 (42.6% of cases)
  decl_exists_1: 37 (36.6% of cases)
  decl_perjury_1: 36 (35.6% of cases)
  decl_calaw_1: 36 (35.6% of cases)
  decl_pkm_1: 35 (34.7% of cases)
  decl_pkm_3: 25 (24.8% of cases)
  decl_

,case_number,disagreement_count,annotator_1,annotator_2,round_1,round_2
94,24NWLC51041,23,Parker,Victor,2,1
78,24NWLC28029,22,Parker,Victor,2,1
69,24NWLC18994,22,Parker,Victor,2,1
95,24NWLC51572,18,Parker,Victor,2,1
48,24CHLC36933,18,Brian,Parker,1,2
89,24NWLC44200,15,Parker,Victor,2,1
16,24CHLC10482,14,Brian,Parker,1,2
14,24CHLC09761,14,Brian,Parker,1,2
28,24CHLC20761,14,Brian,Parker,1,2
71,24NWLC20213,14,Parker,Victor,2,1



SAMPLE DETAILED DISAGREEMENTS (first 50)


,case_number,field,annotator_1,round_1,value_1,annotator_2,round_2,value_2
0,24CHLC00247,rfdj_demand_amount,Brian,2,2899.3,Victor,1,2889.3
1,24CHLC00247,alleged_chargeoff_balance_4,Brian,2,2899.3,Victor,1,2889.3
2,24CHLC00247,ownership_chain_account_match,Brian,2,yes,Victor,1,no
3,24CHLC00247,substantiated_last_payment_date,Brian,2,6/16/22/,Victor,1,06/16/2022
4,24CHLC00758,alleged_last_payment_date_5,Brian,1,09/15/2018,Parker,2,09/15/2022
5,24CHLC00758,hascontractorlaststatement_0,Brian,1,,Parker,2,yes
6,24CHLC00758,decl_exists_1,Brian,1,yes,Parker,2,no
7,24CHLC00758,decl_pkm_1,Brian,1,yes,Parker,2,no
8,24CHLC00758,decl_perjury_1,Brian,1,yes,Parker,2,no
9,24CHLC00758,decl_calaw_1,Brian,1,yes,Parker,2,no



✅ Results saved to /Users/othmanbensouda/Desktop/debt_collection_website/analysis_disagreement/
   - disagreement_summary.xlsx: Cases ranked by disagreement count
   - detailed_disagreements.xlsx: Field-level disagreements with values
   - disagreement_matrix.xlsx: Pivot view (cases × fields)

CASES NEEDING REVIEW (10+ disagreements)


,case_number,disagreement_count,annotator_1,annotator_2,round_1,round_2
94,24NWLC51041,23,Parker,Victor,2,1
78,24NWLC28029,22,Parker,Victor,2,1
69,24NWLC18994,22,Parker,Victor,2,1
95,24NWLC51572,18,Parker,Victor,2,1
48,24CHLC36933,18,Brian,Parker,1,2
89,24NWLC44200,15,Parker,Victor,2,1
16,24CHLC10482,14,Brian,Parker,1,2
14,24CHLC09761,14,Brian,Parker,1,2
28,24CHLC20761,14,Brian,Parker,1,2
71,24NWLC20213,14,Parker,Victor,2,1



⚠️ 38 cases need immediate review

FIELD CATEGORIES WITH MOST DISAGREEMENTS
  Declarations: 524 disagreements (63.9%)
  Other: 109 disagreements (13.3%)
  Allegations: 91 disagreements (11.1%)
  Attorney Fees: 49 disagreements (6.0%)
  RFDJ Amounts: 25 disagreements (3.0%)
  Ownership Chain: 15 disagreements (1.8%)
  Final Recommendation: 7 disagreements (0.9%)
